# Poli: data pipeline

Rebuilds every file in `data/` that the page reads, from the raw rasters and tables in `raw/`.
Run from the repository root. Needs `rasterio`, `geopandas`, `shapely`, `pillow`, `pandas`, `numpy`.

1. **Terrain**: SRTM 90 m v4.1 (Bangladesh, CGIAR-CSI) inside the border and ETOPO5 outside, warped onto one 1024 × 1148 grid over lon 86.5–93.5, lat 19.8–27.0 (about 0.7 km a texel), packed as terrain-RGB so the browser gets 0.1 m precision from an ordinary PNG. Water and sea masks from the WFP/LGED river polygons and GSHHS. GPW v4 2015 population summed into the same texels. An exposure table (land area and people at or below each 0.25 m step to 10 m) from the full-resolution SRTM and the 30 arc-second population grid.
2. **Storms**: IBTrACS v04r01 North Indian basin, every storm since 1960 that came near the window and reached 34 kt, with a hand-curated note for the fourteen the story names.
3. **Land cover**: ESA WorldCover 2021 majority class per texel at 0.35 km, water merged from WorldCover, the WFP/LGED river polygons and OSM 2017, the Hansen 2001–2023 tree-cover-loss share per texel in the Chattogram Hill Tracts, HydroRIVERS reaches (discharge ≥ 150 m³/s) plus the minor Global Map network, names anchored from FFWC gauges, and the class areas.
4. **Checks**: the numbers the page quotes are recomputed here and must match.

In [1]:
import os, sys, json, runpy
from pathlib import Path
REPO = Path.cwd() if (Path.cwd()/"raw").exists() else Path.cwd().parent
os.chdir(REPO); print("repo:", REPO)

repo: /home/claude/work/delta/repo


## 1. Terrain, masks, population, exposure

In [2]:
runpy.run_path(str(REPO/'pipeline'/'make_terrain.py'), run_name='__main__');

grid 1024 x 1148, texel ~0.70 km


srtm cells 296841 (25.3% of grid); h range -1641..2278


water 25.5%, of which river polygons 20.2%


GPW 2015 population on grid: 157.1 M


  <=  1.0 m :     2708 km2 ( 1.9%)    0.7 M people ( 0.4%)
  <=  2.0 m :     4414 km2 ( 3.1%)    1.4 M people ( 0.9%)
  <=  3.0 m :     7390 km2 ( 5.3%)    3.0 M people ( 1.9%)
  <=  5.0 m :    18671 km2 (13.3%)   11.0 M people ( 7.0%)
  <= 10.0 m :    55532 km2 (39.6%)   64.4 M people (41.0%)
done


## 2. Storm tracks

In [3]:
runpy.run_path(str(REPO/'pipeline'/'make_storms.py'), run_name='__main__');

90 storms since 1960 near the window with >= 34 kt
  remapped Aila -> 2009143N17089
  remapped Roanu -> 2016138N10081
  remapped Mora -> 2017147N14087
  remapped Fani -> 2019116N02090
  remapped Yaas -> 2021143N15090
  remapped Sitrang -> 2022295N13093
wrote 90 storms, 1972 points; named notes: ['Bhola', 'Gorky', 'Sidr', 'Aila', 'Mahasen', 'Roanu', 'Mora', 'Fani', 'Bulbul', 'Amphan', 'Yaas', 'Sitrang', 'Mocha', 'Remal']


## 3. Land cover, water, rivers, forest loss

In [4]:
runpy.run_path(str(REPO/'pipeline'/'make_cover.py'), run_name='__main__');

land cover shares %: {'tree cover': 39.24, 'shrubland': 0.0, 'grassland': 0.95, 'cropland': 48.46, 'built-up': 1.07, 'bare': 0.4, 'water': 5.96, 'herbaceous wetland': 1.23, 'mangroves': 2.68}


water texels: worldcover 69622, wfp 951073, osm 115170, merged 1074520


texels with any forest loss in CHT: 92005 mean loss share where >0: 0.076


cover_rgb.png (2296, 2048, 3)


HydroRIVERS reaches 2489, named by anchor 589
rivers.json: 2489 major, 1070 minor


cover_stats: {'sundarbans_km2': 3965, 'hakaluki_km2': 404, 'tanguar_km2': 159, 'cover_shares_pct': {'tree cover': 39.24, 'shrubland': 0.0, 'grassland': 0.95, 'cropland': 48.46, 'built-up': 1.07, 'bare': 0.4, 'water': 5.96, 'herbaceous wetland': 1.23, 'mangroves': 2.68}, 'cover_area_km2': {'tree cover': 54538, 'shrubland': 2, 'grassland': 1326, 'cropland': 67351, 'built-up': 1490, 'bare': 562, 'water': 8286, 'herbaceous wetland': 1704, 'mangroves': 3728}, 'cht_loss_texels': 92005}


## 4. Checks

In [5]:
from PIL import Image
import numpy as np
e = json.load(open(REPO/"data/exposure.json"))
i10, i5 = e["steps_m"].index(10.0), e["steps_m"].index(5.0)
share10 = e["area_km2"][i10] / e["land_total_km2"]; share5 = e["area_km2"][i5] / e["land_total_km2"]
print(f"land <= 10 m: {share10*100:.1f}%   land <= 5 m: {share5*100:.1f}%   people <= 10 m: {e['people'][i10]/1e6:.1f} M of {e['people_total']/1e6:.1f} M")
assert 0.35 < share10 < 0.45 and 0.10 < share5 < 0.17, "exposure shares drifted; check the DEM"
assert 150e6 < e["people_total"] < 165e6, "GPW total should be about 157 M for 2015"
h = np.asarray(Image.open(REPO/"data/height.png")).astype(np.int64)
H = -10000 + (h[...,0]*65536 + h[...,1]*256 + h[...,2]) * 0.1
m = np.asarray(Image.open(REPO/"data/mask.png"))
inside = m[...,0] > 127
print(f"height grid {H.shape}, inside-Bangladesh range {H[inside].min():.0f} .. {H[inside].max():.0f} m, outside {H[~inside].min():.0f} .. {H[~inside].max():.0f} m")
assert H.shape == (1148, 1024) and H[inside].max() > 800 and H[inside].max() < 1100, "SRTM max should be near Keokradong (about 1,000 m)"
s = json.load(open(REPO/"data/storms.json"))
names = {x["note"]["label"] for x in s if "note" in x}
assert {"Bhola", "Gorky", "Sidr", "Aila", "Amphan", "Mocha", "Remal"} <= names, names
print(len(s), "storms;", len(names), "named")
cs = json.load(open(REPO/"data/cover_stats.json"))
sh = cs["cover_shares_pct"]; assert abs(sum(sh.values()) - 100) < 0.5, sh
assert 40 < sh["cropland"] < 55 and 30 < sh["tree cover"] < 45 and 2 < sh["mangroves"] < 3.5, sh
assert 3500 < cs["sundarbans_km2"] < 4500, cs["sundarbans_km2"]
rv = json.load(open(REPO/"data/rivers.json")); assert len(rv["major"]) > 2000 and len(rv["minor"]) > 900
assert len({l["name"] for l in rv["major"] if l.get("name")}) >= 20, "too few named rivers"
c = np.asarray(Image.open(REPO/"data/cover.png")); assert c.shape[:2] == (2296, 2048)
print(f"cover shares: {sh}; Sundarbans {cs['sundarbans_km2']} km2; {len(rv['major'])} major reaches, {len({l['name'] for l in rv['major'] if l.get('name')})} river names")
print("all checks passed")

land <= 10 m: 39.6%   land <= 5 m: 13.3%   people <= 10 m: 64.4 M of 157.1 M
height grid (1148, 1024), inside-Bangladesh range -13 .. 948 m, outside -1640 .. 2278 m
90 storms; 14 named


cover shares: {'tree cover': 39.24, 'shrubland': 0.0, 'grassland': 0.95, 'cropland': 48.46, 'built-up': 1.07, 'bare': 0.4, 'water': 5.96, 'herbaceous wetland': 1.23, 'mangroves': 2.68}; Sundarbans 3965 km2; 2489 major reaches, 21 river names
all checks passed
